# Thử nghiệm bổ sung: Structural Transformer + MixStyle

Thí nghiệm hậu nghiệm, một cấu hình cố định, W64, nguồn duy nhất cho mỗi set.
12 fit mới; tái sử dụng 24 baseline; không đổi kết quả chính của khóa luận.
MixStyle theo lớp trên chuỗi là bản thích ứng đề xuất, chưa phải tái lập bài CNN.
Phải đọc kết quả cùng FAR, không kết luận từ recall/AP đơn lẻ.

Nguồn: [Zhou et al., ICLR 2021](https://arxiv.org/abs/2104.02008).
Xem `code/notes/PHUONG_PHAP_BO_SUNG_MIXSTYLE_2026-11-09.txt`.


In [1]:
from pathlib import Path
import sys, subprocess, json
ROOT = Path.cwd().resolve()
assert (ROOT / 'code/scripts/run_mixstyle_study.py').is_file()
OUT = ROOT / 'code/results/mixstyle_posthoc_2026-11-09'
OUT.mkdir(parents=True, exist_ok=True)
import torch
print('Python:', sys.executable)
assert torch.cuda.is_available(), 'CUDA required'
print('GPU:', torch.cuda.get_device_name(0))
result = subprocess.run([sys.executable, '-m', 'pytest', 'code/tests/test_mixstyle.py', '-q',
                         '--junitxml=' + str(OUT/'mixstyle_tests.xml')],
                        cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
result.check_returncode()


Python: F:\00_Research\.venv\Scripts\python.exe
GPU: NVIDIA GeForce RTX 5060


..........                                                               [100%]
10 passed in 6.90s



In [2]:
# Each completed set is hash-verified and resumable; failed attempts are preserved.
command = [sys.executable, '-u', 'code/scripts/run_mixstyle_study.py', '--stage', 'all', '--output', str(OUT)]
with (OUT/'execution.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=ROOT, text=True, encoding='utf-8', errors='replace',
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in process.stdout:
        log.write(line); log.flush()
        print(line, end='', flush=True)
    status = process.wait()
if status:
    raise RuntimeError(f'Study failed with exit code {status}; see execution.log')


VERIFY input hashes and baseline artifacts


TRAIN set_01 42 source windows 80


F:\00_Research\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\transformers\cuda\attention_backward.cu:902.)


  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


epoch 1 loss 0.7103464007377625


epoch 2 loss 0.6695799827575684


... (416 outputs left out)

    if len(pooled) != 4*3*3*4: raise ValueError('Incomplete experimental matrix')


                               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


ValueError: Incomplete experimental matrix


RuntimeError: Study failed with exit code 1; see execution.log

In [ ]:
import pandas as pd
display(pd.read_csv(OUT/'analysis/equal_set.csv'))
print(json.loads((OUT/'analysis/audit.json').read_text()))
print('Primary result preserved. External results require the companion verification report.')
